# Model Evaluation and Interpretation

This notebook covers:
1. Model evaluation on test data
2. Feature importance analysis
3. SHAP explanations
4. Residual analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import joblib
import json

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [2]:
# Load data and model
df = pd.read_csv('../data/processed/ames_processed.csv')
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = joblib.load('../models/best_model.pkl')
print('Model loaded successfully')

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/ames_processed.csv'

## 1. Model Performance

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# MAPE
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print('Model Performance Metrics:')
print('=' * 40)
print(f'RMSE: ${rmse:,.2f}')
print(f'MAE: ${mae:,.2f}')
print(f'R² Score: {r2:.4f}')
print(f'MAPE: {mape:.2f}%')

# Load saved metrics
with open('../models/model_metrics.json', 'r') as f:
    saved_metrics = json.load(f)
print(f'\nModel: {saved_metrics["model_name"]}')
print(f'Features: {saved_metrics["n_features"]}')

## 2. Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Sale Price ($)')
axes[0].set_ylabel('Predicted Sale Price ($)')
axes[0].set_title('Actual vs Predicted Sale Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Sale Price ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/residual_plots.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Try to get feature importance
try:
    # For tree-based models
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        feature_names = X.columns
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importance
        }).sort_values('Importance', ascending=False)
        
        print('Feature Importance from model:')
        print(importance_df.head(20))
    else:
        # Use permutation importance
        print('Calculating permutation importance...')
        result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
        importance = result.importances_mean
        feature_names = X.columns
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importance
        }).sort_values('Importance', ascending=False)
        
        print(importance_df.head(20))
except Exception as e:
    print(f'Could not calculate feature importance: {e}')

# Save feature importance
importance_df.to_csv('../models/feature_importance.csv', index=False)
print('\nFeature importance saved to models/feature_importance.csv')

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(12, 10))

top_features = importance_df.head(20)
ax.barh(top_features['Feature'], top_features['Importance'])
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. SHAP Analysis

In [ ]:
try:
    import shap
    
    # For tree-based models
    if hasattr(model, 'feature_importances_'):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test[:100])
        
        # Summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_test[:100], show=False)
        plt.tight_layout()
        plt.savefig('../reports/figures/shap_summary.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print('SHAP analysis completed and saved')
    else:
        print('SHAP analysis not supported for this model type')
except ImportError:
    print('SHAP not installed. Install with: pip install shap')
except Exception as e:
    print(f'SHAP analysis error: {e}')

## 5. Summary Statistics

In [ ]:
# Prediction errors
errors = y_test - y_pred

print('Prediction Error Statistics:')
print('=' * 40)
print(f'Mean Error: ${errors.mean():,.2f}')
print(f'Std Error: ${errors.std():,.2f}')
print(f'Max Error: ${errors.max():,.2f}')
print(f'Min Error: ${errors.min():,.2f}')
print(f'95% Error Range: ±${1.96 * errors.std():,.2f}')